In [116]:
import sys
# Define the path to the directory containing the module
module_dir = "../Main/RequiredFuntions"

# Append the directory to sys.path
sys.path.append(module_dir)

In [117]:
import time
import json
import hashlib
import threading
import functions as fn
import dataTransfer as DT
import EncryptionDecryption as ED

In [118]:
with open('../Main/ReceivedData/keys.json', 'r') as file:
    keys = json.load(file)

with open('../Main/ReceivedData/datastore.json', 'r') as file:
    user = json.load(file)

with open('../Main/ReceivedData/datastore_device.json', 'r') as file:
    device = json.load(file)

with open('../Main/ReceivedData/primenumber.json', 'r') as file:
    primenumber = json.load(file)

In [119]:
primenumber = primenumber["primenumber"]

In [120]:
IDi = "RajeshDevice1"
IDg = "rajeshGateway"
IDj = "RajeshHomeDevice1"

    #   Step 1

In [121]:
Ni = fn.nonce_gen()

TDi = hashlib.sha256((IDi + keys["user_public"] + str(Ni)).encode()).hexdigest()
TDg = hashlib.sha256((IDg + keys["gateway_public"] + str(Ni)).encode()).hexdigest()
TDj = hashlib.sha256((IDj + keys["device_public"] + str(Ni)).encode()).hexdigest()

In [122]:
Ci = hashlib.sha256(
    (
        TDi + TDg + TDj + str(Ni)
    ).encode()
).hexdigest()

In [123]:
fingerprintImagePath = "../Main/RequiredData/Fingerprint/11.jpg"
accelerometerDataPath = "../Main/RequiredData/Accelerometer/testingdataM55.csv"

hashI = fn.hash_file(fingerprintImagePath)

hashA = fn.hash_file(accelerometerDataPath)


encryptedImage = ED.symmetric_key_encryption('', 
                                                  ED.read_image_as_bytes(fingerprintImagePath), 
                                                  keys["key"])

encryptedAccelerometerData = ED.symmetric_key_encryption('',
                                                              ED.read_image_as_bytes(accelerometerDataPath), 
                                                              keys["key"])

In [124]:
first_send = {
    "TDi" : TDi,
    "TDg" : TDg,
    "TDj" : TDj,
    "hashI" : hashI,
    "hashA" : hashA,
    "Ni" : Ni,
    "Ci" : Ci,
    "encryptedImage" : encryptedImage,
    "encryptedAccelerometerData" : encryptedAccelerometerData
}

    #   step2

In [125]:
Md = device[IDg][IDj]["partialSecretIntegrityUser"]
Mu = user[IDg][IDi]["partialSecretIntegrityUser"]

In [126]:
Ng = fn.nonce_gen()

In [127]:
Ng

681

In [128]:
M = fn.xor_strings(Mu, Md)
M = fn.xor_strings(M, str(Ng))
M

'd47'

In [129]:
user[IDg][IDi]["gatewayShare"]

17

In [130]:
Pu = Ng ^ user[IDg][IDi]["gatewayShare"]
Pu = fn.xor_strings(str(Pu), IDi)
Pu

'dX\\'

In [131]:
Pd = Ng ^ device[IDg][IDj]["gatewayShare"]
Pd = fn.xor_strings(str(Pd), IDj)
Pd

'eQZ'

In [132]:
Ngi = fn.nonce_gen()
Ngj = fn.nonce_gen()

In [133]:
hashvaluej = hashlib.sha256(
    (
        M + Pd + str(Ngj) +IDj
    ).encode()
).hexdigest()

hashvaluei = hashlib.sha256(
    (
        M + Pu + str(Ngi) + IDi
    ).encode()
).hexdigest()

In [134]:
print(f"hashvaluei = {hashvaluei}")
print(f"hashvaluej = {hashvaluej}")

hashvaluei = 2f6839b380115e6c06173d994004b3c958b3c81d641c303094dca595fd6a4c23
hashvaluej = b15e9dfecd2fa1964f6f2eb9ce2133650f23882a912c73e3d756601ca0e63659


In [135]:
Cj = fn.xor_strings(hashvaluej, str(device[IDg][IDj]["gatewayShare"]))
print(f"Cj = {Cj}")

Ci = fn.xor_strings(hashvaluei, str(user[IDg][IDi]["gatewayShare"]))
print(f"Ci = {Ci}")

Cj = P 
Ci = Q


    #   device

In [136]:
device_message = {
    "M": M,
    "Pd": Pd,
    "Ngj": Ngj,
    "Cj": Cj
}

In [137]:
device_message

{'M': 'd47', 'Pd': 'eQZ', 'Ngj': 347, 'Cj': 'P\x00'}

In [138]:
device_hashCompute = hashlib.sha256(
    (
        device_message["M"] + device_message["Pd"] + str(device_message["Ngj"]) + IDj
    ).encode()
).hexdigest()

device_hashCompute

'b15e9dfecd2fa1964f6f2eb9ce2133650f23882a912c73e3d756601ca0e63659'

In [139]:
Sgj = fn.xor_strings(device_message["Cj"], device_hashCompute)

Sgj = int(Sgj)
Sgj

21

In [140]:
device[IDg][IDj]["userShare"]

49

In [141]:
device_hashCompute == hashvaluej

True

In [142]:
Sgj == device[IDg][IDj]["gatewayShare"]

True

In [143]:
Sj = ( (2 * device[IDg][IDj]["userShare"]) - Sgj) % primenumber
Sj

24

In [144]:
hashlib.sha256(
    (
        str(Sj) + IDg
    ).encode()
).hexdigest() == device[IDg][IDj]["partialSecretIntegrityGateway"]

False

In [145]:
Vj = hashlib.sha256(
    ( 
        str(Sj) + IDj
    ).encode()
).hexdigest()

In [146]:
Vj

'd113a81a1a24fa016383765f249d41a13e0bf80f77d292de4511db0cb49e17eb'

In [ ]:
Ng = fn.xor_strings(device_message["Pd"], str(Sgj))
Ng = fn.xor_strings(Ng, IDj)

In [148]:
Ng

'\x05\x01'

In [149]:
device_message

{'M': 'd47', 'Pd': 'eQZ', 'Ngj': 347, 'Cj': 'P\x00'}

In [150]:
Vi = fn.xor_strings(device_message["M"], Vj)
Vi = fn.xor_strings(Vi, str(Ng))

In [151]:
Vi

'\x05\x04'

In [152]:
Ks = hashlib.sha256(
    (
        Vi + str(Ng) + Vj
    ).encode()
).hexdigest()

In [153]:
Ks

'801ce272e42856998a87bc65bc7a02dff5725dca713199782e659e7156ff2bbb'

    #   user

In [154]:
user_message = {
    "M": M,
    "Pu": Pu,
    "Ngi": Ngi,
    "Ci": Ci
}

In [155]:
user_message

{'M': 'd47', 'Pu': 'dX\\', 'Ngi': 76, 'Ci': '\x03Q'}

In [156]:
user_hashCompute = hashlib.sha256(
    (
        user_message["M"] + user_message["Pu"] + str(user_message["Ngi"]) + IDi
    ).encode()
).hexdigest()

user_hashCompute

'2f6839b380115e6c06173d994004b3c958b3c81d641c303094dca595fd6a4c23'

In [157]:
Sgi = fn.xor_strings(user_message["Ci"], user_hashCompute)
Sgi = int(Sgi)
Sgi

17

In [158]:
user[IDg][IDi]["userShare"]

48

In [159]:
user_hashCompute == hashvaluei

True

In [160]:
user[IDg][IDi]["gatewayShare"] == Sgi

True

In [161]:
Si = ( -Sgi + (2 * user[IDg][IDi]["userShare"])) % primenumber
Si

26

In [162]:
hashlib.sha256(
    (
        str(Si) + IDg
    ).encode()
).hexdigest()

'fd68334dc06725fd08c160fd05879f7ba91b1198fc916b2438300600d980873e'

In [163]:
hashlib.sha256(
    (
        str(Si) + IDg
    ).encode()
).hexdigest() == user[IDg][IDi]["partialSecretIntegrityGateway"]

False

In [164]:
Vi = hashlib.sha256(
    (
        str(820) + IDg
    ).encode()
).hexdigest()

In [165]:
Vi

'9dee589983aff3536433360a2358087c31279541bb74c005e3c78adf907c0c7f'

In [166]:
Vi == user[IDg][IDi]["partialSecretIntegrityGateway"]

False

In [167]:
Ng = fn.xor_strings(user_message["Pu"], str(820))
Ng = fn.xor_strings(Ng, IDi)

In [168]:
Ng

'\x0e\x0b\x06'

In [169]:
Vj = fn.xor_strings(user_message["M"], Vi)
Vj = fn.xor_strings(Vj, str(Ng))

In [170]:
Vj

'S[T'

In [171]:
Ksj = hashlib.sha256(
    (
        Vi + Ng + Vj
    ).encode()
).hexdigest()

In [172]:
Ks == Ksj

False